# Guitar Stem Separation — `htdemucs-6s-guitar-ft`

This notebook separates guitar (and optionally all other stems) from a song using [adityalakhani/htdemucs-6s-guitar-ft](https://huggingface.co/adityalakhani/htdemucs-6s-guitar-ft), a guitar-specialized fine-tune of Meta AI's `htdemucs_6s`.

**How to use on Kaggle:**
1. Turn on a **GPU accelerator** (Settings → Accelerator → GPU T4 x2 or P100). CPU works but is much slower.
2. Add your audio file(s) as a Kaggle Dataset input, or just upload directly under `/kaggle/working/` via the notebook's file browser, or download a file from a URL in the cell below.
3. Set `AUDIO_PATH` in the config cell, then Run All. **mp4/mov/video files work fine** — the notebook uses `ffmpeg` to pull out just the audio track before separation.
4. Output stems are written to `/kaggle/working/output/` and zipped at the end so you can download them from the notebook's Output tab.

6 stems the model outputs (in order): `drums`, `bass`, `other`, `vocals`, `guitar` (index 4), `piano`.

## 1. Install dependencies

In [ ]:
# Kaggle already has torch + torchaudio preinstalled with CUDA.
# demucs 4.0.1 matches what the model card was trained/tested against.
!pip install -q demucs==4.0.1 huggingface_hub soundfile


## 2. Config

Set `AUDIO_PATH` to a file already on disk, or set `AUDIO_URL` to download one first. Leave `AUDIO_URL` as `None` if you're pointing directly at a file (e.g. from an attached Kaggle Dataset under `/kaggle/input/...`).

In [ ]:
import os

# --- EDIT THESE ---
AUDIO_URL = None                                   # e.g. "https://example.com/song.mp3", or None
AUDIO_PATH = "/kaggle/working/input/song.mp4"       # path to the local audio/video file (mp3, mp4, wav, mov, etc. all fine)
OUTPUT_DIR = "/kaggle/working/output"

SEPARATE_ALL_STEMS = True   # False = only export the guitar stem (faster to write, smaller output)
SHIFTS = 1                  # higher = better quality, slower (demucs test-time augmentation)
OVERLAP = 0.25              # overlap between chunks during inference
# -------------------

os.makedirs(os.path.dirname(AUDIO_PATH), exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

if AUDIO_URL:
    import urllib.request
    print(f"Downloading {AUDIO_URL} -> {AUDIO_PATH}")
    urllib.request.urlretrieve(AUDIO_URL, AUDIO_PATH)

assert os.path.exists(AUDIO_PATH), (
    f"No file found at {AUDIO_PATH}. Either set AUDIO_URL to download one, "
    f"or point AUDIO_PATH at a file you've uploaded / attached as a Kaggle Dataset."
)
print("Input audio:", AUDIO_PATH)


## 3. Extract audio (handles mp4 / video files too)

Runs the input through `ffmpeg` regardless of format — this covers mp4, mov, mkv, m4a, mp3, etc. and normalizes everything to a clean 44.1kHz stereo WAV before separation. Kaggle's base image ships with `ffmpeg` already installed.

In [ ]:
import subprocess

EXTRACTED_WAV = "/kaggle/working/input/_extracted_audio.wav"
os.makedirs(os.path.dirname(EXTRACTED_WAV), exist_ok=True)

cmd = [
    "ffmpeg", "-y", "-i", AUDIO_PATH,
    "-vn",                 # drop any video stream (mp4/mov/etc.)
    "-ar", "44100",
    "-ac", "2",
    EXTRACTED_WAV,
]
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise RuntimeError("ffmpeg failed to extract audio — see log above.")

SEPARATION_INPUT = EXTRACTED_WAV
print("Extracted audio ->", SEPARATION_INPUT)


## 4. Check GPU

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)
if DEVICE == "cuda":
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(name, "- compute capability", cap)
    supported = torch.cuda.get_arch_list()
    sm = f"sm_{cap[0]}{cap[1]}"
    if sm not in supported:
        print(
            f"\nWARNING: this GPU is {sm}, but the installed PyTorch only supports "
            f"{supported}. Kaggle's P100 accelerator (sm_60) is too old for the "
            f"preinstalled PyTorch build. Go to Settings > Accelerator and switch "
            f"to 'GPU T4 x2' (or any T4/A100 option) instead of P100, then Restart "
            f"& Run All. Falling back to CPU for now so this cell doesn't crash.\n"
        )
        DEVICE = "cpu"
else:
    print("No GPU detected — enable one under Settings > Accelerator for much faster inference.")


## 5. Download the fine-tuned checkpoint from Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download

checkpoint_path = hf_hub_download(
    repo_id="adityalakhani/htdemucs-6s-guitar-ft",
    filename="guitar_htdemucs_6s.pt",
)
print("Checkpoint downloaded to:", checkpoint_path)


## 6. Load the model and fine-tuned weights

In [ ]:
from demucs.pretrained import get_model

STEM_NAMES = ["drums", "bass", "other", "vocals", "guitar", "piano"]
GUITAR_STEM_INDEX = STEM_NAMES.index("guitar")

# Base htdemucs_6s architecture (downloads Meta AI's pretrained weights first, then we overwrite them)
model = get_model("htdemucs_6s")

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
state_dict = checkpoint["model_state_dict"]

# get_model() wraps the network in a BagOfModels — remap keys to match
state_dict = {f"models.0.{k}": v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()
model.to(DEVICE)

print("Model loaded. Stems:", STEM_NAMES)


## 7. Load audio

In [ ]:
import soundfile as sf
import numpy as np

SAMPLE_RATE = 44100

# SEPARATION_INPUT was already normalized to 44.1kHz stereo WAV by ffmpeg in step 3,
# so this is just a plain, reliable soundfile read (works the same for an mp4 source,
# an mp3, a wav, etc. — ffmpeg already handled the format-specific decoding).
audio_np, sr = sf.read(SEPARATION_INPUT, always_2d=True, dtype="float32")
waveform = torch.from_numpy(audio_np.T)  # (channels, time)

if waveform.shape[0] == 1:
    waveform = waveform.repeat(2, 1)  # mono -> stereo
elif waveform.shape[0] > 2:
    waveform = waveform[:2]  # drop extra channels

print("Waveform shape (channels, samples):", waveform.shape)
print(f"Duration: {waveform.shape[1] / SAMPLE_RATE:.1f}s")


## 8. Run separation

Uses `apply_model` for proper overlap-add chunking, so this works on full-length songs regardless of GPU memory.

In [ ]:
from demucs.apply import apply_model
import time

start = time.time()
with torch.no_grad():
    sources = apply_model(
        model,
        waveform.unsqueeze(0).to(DEVICE),
        shifts=SHIFTS,
        overlap=OVERLAP,
        progress=True,
    )
sources = sources[0].cpu()  # (stems, channels, time)
print(f"Separation finished in {time.time() - start:.1f}s")


## 9. Save stems

In [ ]:
track_name = os.path.splitext(os.path.basename(AUDIO_PATH))[0]
track_out_dir = os.path.join(OUTPUT_DIR, track_name)
os.makedirs(track_out_dir, exist_ok=True)

stems_to_save = STEM_NAMES if SEPARATE_ALL_STEMS else ["guitar"]

for name in stems_to_save:
    idx = STEM_NAMES.index(name)
    stem_audio = sources[idx]  # (channels, time)
    out_path = os.path.join(track_out_dir, f"{name}.wav")
    sf.write(out_path, stem_audio.T.numpy(), samplerate=SAMPLE_RATE, subtype="PCM_24")
    print("Saved:", out_path)

print("\nDone. Stems written to:", track_out_dir)


## 10. Listen to a stem (optional, in-notebook playback)

In [ ]:
from IPython.display import Audio, display

guitar_path = os.path.join(track_out_dir, "guitar.wav")
if os.path.exists(guitar_path):
    print("Guitar stem:")
    display(Audio(guitar_path))


## 11. Zip the output for easy download

In [ ]:
import shutil

zip_base = os.path.join(OUTPUT_DIR, f"{track_name}_stems")
zip_path = shutil.make_archive(zip_base, "zip", track_out_dir)
print("Zipped output:", zip_path)
print("Find it in the notebook's Output tab (top right) once the run finishes.")
